# Lesson 04 - ടൂൾ ഉപയോഗ ഡിസൈൻ പാറ്റേൺ

ഈ പാഠത്തിൽ നിങ്ങൾ മൈക്രോസോഫ്റ്റ് ഏജന്റ് ഫ്രെയിംവർക്ക് (Python) ഉപയോഗിച്ച് AI ഏജന്റുകൾക്കുള്ള **ടൂൾ ഉപയോഗ** ഡിസൈൻ പാറ്റേൺ പഠിക്കും. നാം ഇതിൽ ഉൾപ്പെടുന്നു:

- `@tool` ഡെകോറെറ്ററുമായി ഫംഗ്ഷൻ ടൂളുകൾ നിർവചിക്കുകയും ടൈപ്പുചെയ്ത പാരാമീറ്ററുകളും
- മോഡൽ ഓരോ ടൂളും ചെയ്യുന്നത് അറിയാൻ ടൂൾ സ്കീമകൾ നൽകൽ
- `approval_mode` ഉപയോഗിച്ച് ടൂൾ എക്സിക്യൂഷൻ നിയന്ത്രിക്കൽ
- Pydantic മോഡലുകളും `response_format` ഉപയോഗിച്ച് **സംരചിത ഔട്ട്‌പുട്ട്** മടങ്ങിവരുത്തൽ

സന്നിവേശം ഒരു **ട്രാവൽ ബുക്കിംഗ് ഏജന്റ്** ആണ്, ഇത് ലക്ഷ്യസ്ഥാനങ്ങൾ പരിശോധിക്കാനും, ലഭ്യത പരിശോധിക്കാനും, വിമാന വിവരങ്ങൾ നേടാനുമാകും.


## ക്രമീകരണം


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool ഡെക്കറേറ്റർ ഉപയോഗിച്ച് ടൂൾসുകൾ നിർവചിക്കൽ

`@tool` ഡെക്കറേറ്റർ ഒരു സാധാരണ Python ഫംഗ്ഷനിനെ ഏജന്റ് വിളിക്കാവുന്ന ഒരു ടൂളായി മാറ്റുന്നു.
പ്രധാനപ്പെട്ട نقاط:

- **ഡോക്സ്ട്രിംഗ്** ടൂളിന്റെ വിവരണമായാണ് മോഡൽ കാണുന്നത്.
- **ടൈപ്പ് ആനോട്ടേഷനുകൾ** (വിവരണങ്ങളോടെയുള്ള `Annotated` ഉൾപ്പെടെ) ടൂൾ സ്കീമ നിർവചിക്കുന്നു.
- `approval_mode` ഓരോ കോളും പ്രവർത്തിക്കാൻ മുമ്പ് ഉപയോക്താവ് അനുമതി നൽകണമെന്ന് നിയന്ത്രിക്കുന്നു.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## ഒന്നിലധികം ടൂളുകളുള്ള ഏജന്റ് സൃഷ്ടിക്കൽ

ഉപഭോക്താവിന്റെ ചോദ്യത്തിന് ഉത്തരം നൽകുന്നതിന് ആവശ്യമുള്ള ഏതു ടൂളും മോഡൽ ഉപയോഗിക്കാവുന്നതാണ് എന്നതിനായി മൂന്ന് ടൂളുകളും ക്ലയന്റിലേക്ക് നൽകുക.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## ടൂളുകളുമായി ഘടിത ഔട്ട്‌പുട്ട്

`response_format` ഒരു Pydantic മോഡലാക്കി സജ്ജീകരിക്കുന്നതിലൂടെ, ഏജൻറ് സ്വതന്ത്ര രൂപത്തിലുള്ള എഴുത്തി പകരം സുതാര്യമായി ടൈപ്പ് ചെയ്ത JSON ഓബ്ജക്റ്റ് മറുപടി നൽകാൻ നൃബന്ധിതനാകുന്നു. ഇത് ഡൗൺസ്ട്രീം കോഡ് ഫലത്തെ പ്രോഗ്രാമാറ്റിക്കായി ഉപയോഗിക്കേണ്ടതിനുള്ളപ്പോൾ ഉപകാരപ്രദമാണ്.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## ടൂൾ അപ്രൂവൽ പാറ്റേണുകൾ

`@tool`-ലെ `approval_mode` പാരാമീറ്റർ ടൂൾ കോൾകൾ പ്രവർത്തിപ്പിക്കുന്നതിനു മുൻപ് മനുഷ്യ അനുമതി ആവശ്യമാണ് എന്നത് നിയന്ത്രിക്കുന്നു:

| മോഡ് | പെരുമാറ്റം |
|---|---|
| `"never_require"` | ടൂൾ സ്വയം പ്രവർത്തിക്കുന്നു — ഉപയോക്തൃ സ്ഥിരീകരണം ആവശ്യമില്ല. |
| `"always_require"` | എല്ലാ കോൾക്കും പ്രവർത്തിപ്പിക്കുന്നതിനുമുമ്പ് ഉപയോക്തൃ അംഗീകാരം കമ്മിടണം. |

സൈഡ്-എഫക്റ്റ് ഉണ്ടാക്കുന്ന (ഉദാ: ടിിക്കറ്റ് ബുക്കിംഗ്, ക്രെഡിറ്റ് കാർഡ് ചാർജിംഗ് തുടങ്ങിയവ) ടൂളുകൾക്ക് `"always_require"` ഉപയോഗിക്കുക, അതിനാൽ മനുഷ്യൻ സൈക്കിളിൽ തുടരും.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## സാരാംശം

ഈ പാഠത്തിൽ നിങ്ങൾ പഠിച്ചത്:

1. ടൂൾ സ്കീമയായി പ്രവർത്തിക്കുന്ന ടൈപ്പുള്ള പാരാമീറ്ററുകളും ഡോക്സ്ട്രിംഗുകളും ഉപയോഗിച്ച് `@tool` ഡെക്കറേറ്റർ ഉപയോഗിച്ച് **ടൂളുകൾ നിർവ്വചിക്കുന്നത്**.
2. ഏജന്റ് പരസ്പരം സങ്കീർണമായ ചോദ്യങ്ങൾക്ക് മറുപടി നൽകാൻ അനുക്രമത്തിൽ വിളിക്കാനാകുന്ന många tools** കോമ്പോസ് ചെയ്യുക.
3. `response_format` ആയി Pydantic മോഡൽ പാസ്സ് ചെയ്തു **സംഘടിത ഔട്ട്പുട്ട് തിരികെ ലഭിക്കണം**.
4. **സേഫ് ഓപ്പറേഷനുകൾക്കായി മനുഷ്യനെ ഇടയിൽ ഉൾക്കൊള്ളിക്കാൻ** `approval_mode` ഉപയോഗിച്ച് ടൂൾ അംഗീകാര നിയന്ത്രണം.

ഈ രീതികൾ പുറമേത്തുള്ള സംവിധാനങ്ങളുമായി സുരക്ഷിതമായി ഇടപഴകാൻ കഴിയുന്ന വിശ്വസനീയവും പ്രോഡക്ഷൻ-സജ്ജവുമായ ഏജന്റിനെ രൂപപ്പെടുത്തുന്നതിനുള്ള അടിസ്ഥാനം ആണ്.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**വ്യക്തമായ കിഴിവ്**:  
ഈ ഡോക്യുമെന്റ് AI വിവർത്തന സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് വിവർത്തനം ചെയ്തതാണ്. കൃത്യത ഉറപ്പാക്കാൻ ഞങ്ങൾ ശ്രമിക്കുന്നുണ്ടെങ്കിലും, ഓട്ടോമേറ്റഡ് വിവർത്തനങ്ങളിൽ തെറ്റുകൾ അല്ലെങ്കിൽ അശുദ്ധികൾ ഉണ്ടാകാമെന്നത് ദയവായി ശ്രദ്ധിക്കുക. അഭ്യസിക്കപ്പെട്ട ഡോക്യുമെന്റ് അതിന്റെ മൗലിക ഭാഷയിലുള്ളത് ആണ് ഏറ്റവും പ്രാമാണികമായ ഉറവിടം. പ്രധാന വിവരങ്ങൾക്ക് പ്രൊഫഷണൽ മനുഷ്യ വിവർത്തനം ശിപാർശ ചെയ്യപ്പെടുന്നു. ഈ വിവർത്തനത്തിന്റെ ഉപയോഗത്തിൽ നിന്നുണ്ടാകുന്ന ഏതെങ്കിലും തെറ്റിദ്ധാരണകൾക്കും തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കും ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
